# CSA0812 – Assignment 5
## A5 – Design an AI Application for Citizen Complaints

**Student:** Darshan A  
**Register Number:** 1973260011  
**Course Outcome:** CO5  
**Marks:** 4 | K3 – Apply

**Purpose:** This notebook is organised as a separate submission for this rubric item. Written explanations are provided in text cells. Where implementation is required, the Python cells can be run in Google Colab.

## A5.1 – Sentiment and Summary (2 Marks)

### Lexicon-based sentiment scorer

Maintain two word lists:
- Positive words: `good`, `clean`, `fast`, `helpful`, `improved`.
- Negative words: `bad`, `dirty`, `slow`, `rude`, `smoke`, `broken`.

The scorer tokenises a complaint, gives +1 for a positive word and -1 for a negative word, and adds the values. A preceding negator such as `not`, `never` or `no` reverses the value of the following sentiment word. The final score is labelled positive if greater than zero, negative if below zero and neutral if equal to zero.

**Negation examples**

| Complaint | Expected behaviour |
|---|---|
| `The air is good.` | Positive word contributes +1 |
| `The air is not good.` | `not` flips `good` to -1 |
| `The air is not bad.` | `not` flips `bad` to +1 |

This basic rule handles simple negation but does not fully understand sarcasm, long-distance negation, double negatives or mixed opinions.

### Extractive summariser

1. Split the complaint text into sentences.
2. Convert words to lowercase and remove punctuation.
3. Remove common stop words such as `the`, `is`, `and`, `to`.
4. Count the frequency of important remaining words across the complaints.
5. Score each sentence by adding the frequency of its important words.
6. Select the highest-scoring sentences, then return them in their original order.

The summariser selects sentences from the original complaints; it does not create new sentences. That makes it extractive rather than generative.

## A5.2 – Basic Chatbot (2 Marks)

### Rule-based question answering design

The chatbot supports a narrow question such as: “What is the PM2.5 in Adyar today?”

**Processing rules**

1. Convert the question to lowercase and identify whether it asks for PM2.5.
2. Extract a station name from the known station list.
3. Recognise the date keyword `today`, `yesterday` or an explicit YYYY-MM-DD date.
4. If station or date is missing or ambiguous, ask the citizen to clarify.
5. Query SQLite using parameter placeholders, not string concatenation.
6. Return the latest or daily mean PM2.5 value, the station, date and unit.
7. If no reading is available, state that the database has no valid result.

**Example parameterised query**

```sql
SELECT AVG(pm25)
FROM readings
WHERE station_id = ?
  AND observation_date = ?;
```

The station ID and date are passed as parameters. This prevents a citizen's input from being interpreted as SQL code.

### Limitations

- It only understands predefined keywords and question patterns.
- It may not understand spelling mistakes, synonyms or complex natural language.
- It cannot infer a station name that is not in the station list.
- It cannot answer questions that are not covered by the programmed rules.
- It may give no result when readings are missing.
- It should provide a data timestamp and avoid presenting an old reading as current.

## Alternative Concept and Complexity Analysis

A trained Naive Bayes sentiment classifier could be trained on complaints labelled by humans. A pre-trained language model could also be used to interpret context and more complex wording.

| Factor | Word-list method | Trained ML / pre-trained language model |
|---|---|---|
| Time per complaint | Usually O(w), where w is the number of words | Naive Bayes prediction is typically proportional to the features; a language model may require much more computation per token |
| Memory | Very small: lexicon and rule sets | Stores vocabulary, model parameters or model weights; a language model may require substantial memory |
| Accuracy | Transparent and fast, but limited with sarcasm, double negation and mixed meaning | Can use context and learned word patterns; expected performance depends on labelled local data and evaluation |
| Explainability | Easy to show which words contributed to the score | More difficult to explain individual predictions, depending on model |
| Maintenance | Update word lists and rules manually | Collect labelled examples, train/evaluate and monitor model drift |

No accuracy percentage should be assumed without a test set. The board should label a representative sample of local complaints, split it into training and testing sets, and compare precision, recall and accuracy before deploying a trained model. A human review path should remain available for uncertain or high-impact complaints.

## A5 – Conclusion

A lexicon scorer and extractive summariser provide transparent introductory AI features for understanding citizen complaints. A rule-based chatbot can answer common station/date queries by using parameterised SQL. These methods are lightweight but limited in context understanding; a trained model should be considered only after collecting labelled local data and measuring its performance.